# V0.6 Capability Denial Lab

问题：LLM 明明主动要求修改文件，Kernel 为什么可以拒绝？

这个 notebook 逐格展示 LLM intent 与 Kernel authority 的分离。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v06", mode=MODE)


## Step 1: Setup

注册一个写文件 tool，同时创建有权限和无权限两个 Agent。

In [ ]:
lab.setup()

## Step 2: Inspect denied model request

无权限 Agent 的模型请求里不应该出现写工具。

In [ ]:
lab.show_model_request()

## Step 3: Ask deterministic or real model

模型只能基于 Kernel 暴露的工具集合提出行动。

In [ ]:
lab.model_step()

## Step 4: Force an unauthorized ToolCall

即使外部伪造 ToolCall，执行阶段仍会再次授权并拒绝。

In [ ]:
lab.forced_unauthorized_execution()

## Step 5: Authorized comparison

同一个 ToolCall，在有 capability 的 Agent 上可以通过。

In [ ]:
lab.authorized_comparison()

## Summary

这个实验回答：Capability 是 Kernel authority，不是模型自述。

In [ ]:
lab.summary()
lab.close()